In [ ]:
##  %matplotlib inline
%matplotlib widget

import matplotlib as mpl
mpl.rcParams['animation.html'] = 'jshtml'
mpl.rcParams['animation.embed_limit'] = 200.0

import numpy as np
import math
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import happi

In [ ]:
# path = "../work_dir/tuto_3d"
# path = "../work_dir/tuto_3d_SM0"
# path = "../work_dir/tuto_3d_PML3b"
# path = "../work_dir/tuto_3d_PML3b"
# path = "../work_dir/tuto_3d_PML"
# path = "../work_dir/rot_anal/t2"
# path = "../work_dir/rot_anal/t-Gauss"
# path = "../work_dir/benchmark00"
path = "../work_dir/tuto_3d_PML"
# path = "../work_dir/rot_anal/t6"
# path = "../work_dir/rot_anal/y-rot"
# path = "../work_dir/rot_anal/y-rot-z"
# path = "../work_dir/rot_anal/z-rot"
# path = "../work_dir/rot_anal/z-rot-y"
# path = "../work_dir/rot_anal/yz-rot"
# path = "../work_dir/rot_anal/yz-rot-yz"
# path = "../work_dir/rot_anal/t-Gauss2"
# path = "../work_dir/rot_anal/y-rot-z-Sm"
field_component = "Bz"
axes_aspect = 'auto' # 'equal'


S = happi.Open(path, verbose=False)
print(S.namelist.Main.timestep) 
print(S.namelist.Main.geometry)
for species in S.namelist.Species:
    print("species "+species.name+" has mass "+str(species.mass))

Diag = S.Field(0,"Ex")
xxx = Diag.getData()
print(np.shape(xxx))

In [ ]:
%matplotlib widget

import happi
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import IntSlider, SelectionSlider, VBox, HBox
from IPython.display import display

# =========================================================
# USER OPTIONS
# =========================================================

FIELD_FAMILY = "B"        # "B" or "E"
COLORBAR_MODE = "global"  # "global" or "adaptive"

cmap ="RdBu_r" # "viridis" #"RdBu_r"

AXES = ["x", "y", "z"]

# Use 1 for a true scan over all selected timesteps.
# Increase only if the global scan is too expensive.
TIMESTEP_STRIDE = 1

VMAX_FALLBACK = 1.0

# =========================================================
# OPEN SIMULATION
# =========================================================

S = happi.Open(path, verbose=False)

# =========================================================
# FIELD FAMILY / POLARISATIONS
# =========================================================

FIELD_FAMILY = FIELD_FAMILY.upper()
COLORBAR_MODE = COLORBAR_MODE.lower()

if FIELD_FAMILY == "B":
    POLARISATIONS = ["Bx", "By", "Bz"]
    REFERENCE_FIELD = "By"
elif FIELD_FAMILY == "E":
    POLARISATIONS = ["Ex", "Ey", "Ez"]
    REFERENCE_FIELD = "Ey"
else:
    raise ValueError("FIELD_FAMILY must be either 'B' or 'E'.")

if COLORBAR_MODE not in ["global", "adaptive"]:
    raise ValueError("COLORBAR_MODE must be either 'global' or 'adaptive'.")

DEFAULT_FIELD = POLARISATIONS[1]

# =========================================================
# GEOMETRY
# =========================================================

ax_idx = {
    "x": 0,
    "y": 1,
    "z": 2,
}

N_spatial = {}
spatial_values = {}

for AXIS in AXES:

    L = S.namelist.Main.grid_length[ax_idx[AXIS]]
    d = S.namelist.Main.cell_length[ax_idx[AXIS]]

    N = int(round(L / d))

    N_spatial[AXIS] = N
    spatial_values[AXIS] = np.arange(N) * d

# =========================================================
# TIMESTEPS
# =========================================================

F0 = S.Field(0, REFERENCE_FIELD)
timesteps = np.asarray(F0.getTimesteps())
timesteps = timesteps[::TIMESTEP_STRIDE]

if len(timesteps) == 0:
    raise RuntimeError("No timesteps available after applying TIMESTEP_STRIDE.")

# =========================================================
# HELPER FUNCTIONS
# =========================================================

def raw_absmax(data):
    arr = np.asarray(data)

    if arr.size == 0:
        return 0.0

    abs_arr = np.abs(arr)

    if np.all(np.isnan(abs_arr)):
        return 0.0

    val = np.nanmax(abs_arr)

    if not np.isfinite(val):
        return 0.0

    return float(val)


def finalise_vmax(vmax):
    if not np.isfinite(vmax) or vmax <= 0.0:
        return VMAX_FALLBACK

    return float(vmax)


def load_plane(FIELD, t, normal_axis, normal_index):
    """
    normal_axis = "z" gives xy plane
    normal_axis = "y" gives xz plane
    normal_axis = "x" gives yz plane
    """
    coord = spatial_values[normal_axis][normal_index]

    D = S.Field(
        0,
        FIELD,
        timesteps=t,
        subset={normal_axis: [coord]},
    )

    data = np.asarray(D.getData()[0])
    data = np.squeeze(data)

    return data, coord


def plane_extent(horizontal_axis, vertical_axis):
    h = spatial_values[horizontal_axis]
    v = spatial_values[vertical_axis]

    return [
        h[0],
        h[-1],
        v[0],
        v[-1],
    ]


def adaptive_limits(data):
    vmax = finalise_vmax(raw_absmax(data))
    return -vmax, vmax


def fixed_global_limits():
    return -GLOBAL_VMAX, GLOBAL_VMAX


def current_limits(data):
    if COLORBAR_MODE == "global":
        return fixed_global_limits()

    if COLORBAR_MODE == "adaptive":
        return adaptive_limits(data)

    raise ValueError("Unknown COLORBAR_MODE.")

# =========================================================
# GLOBAL MAXIMUM SCAN
# =========================================================

GLOBAL_VMAX = None

if COLORBAR_MODE == "global":

    print("Computing global normalisation...")
    print(f"Field family: {FIELD_FAMILY}")
    print(f"Components: {POLARISATIONS}")
    print(f"Number of timesteps used: {len(timesteps)}")

    vmax_scan = 0.0
    counter = 0
    total = len(POLARISATIONS) * len(timesteps)

    for FIELD in POLARISATIONS:

        for t in timesteps:

            counter += 1

            D = S.Field(
                0,
                FIELD,
                timesteps=t,
            )

            data = D.getData()[0]

            vmax_scan = max(
                vmax_scan,
                raw_absmax(data),
            )

            print(
                f"\rScanning {counter}/{total}: {FIELD}, t={t}",
                end="",
            )

    GLOBAL_VMAX = finalise_vmax(vmax_scan)

    print()
    print(f"GLOBAL_VMAX = {GLOBAL_VMAX}")

else:
    print("Adaptive colourbar mode selected.")
    print("No global scan will be performed.")

# =========================================================
# INITIAL INDICES
# =========================================================

itime0 = 0

ix0 = N_spatial["x"] // 2
iy0 = N_spatial["y"] // 2
iz0 = N_spatial["z"] // 2

t0 = timesteps[itime0]
FIELD0 = DEFAULT_FIELD

# =========================================================
# INITIAL DATA
# =========================================================

field_xy, z_coord = load_plane(
    FIELD=FIELD0,
    t=t0,
    normal_axis="z",
    normal_index=iz0,
)

field_xz, y_coord = load_plane(
    FIELD=FIELD0,
    t=t0,
    normal_axis="y",
    normal_index=iy0,
)

field_yz, x_coord = load_plane(
    FIELD=FIELD0,
    t=t0,
    normal_axis="x",
    normal_index=ix0,
)

vmin_xy, vmax_xy = current_limits(field_xy)
vmin_xz, vmax_xz = current_limits(field_xz)
vmin_yz, vmax_yz = current_limits(field_yz)

# =========================================================
# FIGURE
# =========================================================

plt.ioff()

fig = plt.figure(
    figsize=(18, 6),
    constrained_layout=True,
)

gs = fig.add_gridspec(
    2,
    3,
    height_ratios=[1.0, 0.06],
)

axs = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[0, 2]),
]

caxs = [
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
    fig.add_subplot(gs[1, 2]),
]

fig.suptitle(
    f"{FIELD0} | t index = {itime0}, t = {t0} | "
    f"colourbar mode = {COLORBAR_MODE}",
    fontsize=14,
)

# ---------------------------------------------------------
# xy plane, z fixed
# ---------------------------------------------------------

im_xy = axs[0].imshow(
    field_xy.T,
    extent=plane_extent("x", "y"),
    origin="lower",
    aspect="auto",
    cmap=cmap,
    vmin=vmin_xy,
    vmax=vmax_xy,
)

axs[0].set_title(f"xy plane | z = {z_coord:.4g}")
axs[0].set_xlabel("x")
axs[0].set_ylabel("y")

cbar_xy = fig.colorbar(
    im_xy,
    cax=caxs[0],
    orientation="horizontal",
)
cbar_xy.set_label(FIELD0)

# ---------------------------------------------------------
# xz plane, y fixed
# ---------------------------------------------------------

im_xz = axs[1].imshow(
    field_xz.T,
    extent=plane_extent("x", "z"),
    origin="lower",
    aspect="auto",
    cmap=cmap,
    vmin=vmin_xz,
    vmax=vmax_xz,
)

axs[1].set_title(f"xz plane | y = {y_coord:.4g}")
axs[1].set_xlabel("x")
axs[1].set_ylabel("z")

cbar_xz = fig.colorbar(
    im_xz,
    cax=caxs[1],
    orientation="horizontal",
)
cbar_xz.set_label(FIELD0)

# ---------------------------------------------------------
# yz plane, x fixed
# ---------------------------------------------------------

im_yz = axs[2].imshow(
    field_yz.T,
    extent=plane_extent("y", "z"),
    origin="lower",
    aspect="auto",
    cmap=cmap,
    vmin=vmin_yz,
    vmax=vmax_yz,
)

axs[2].set_title(f"yz plane | x = {x_coord:.4g}")
axs[2].set_xlabel("y")
axs[2].set_ylabel("z")

cbar_yz = fig.colorbar(
    im_yz,
    cax=caxs[2],
    orientation="horizontal",
)
cbar_yz.set_label(FIELD0)

# =========================================================
# SLIDERS
# =========================================================

slider_time = IntSlider(
    min=0,
    max=len(timesteps) - 1,
    step=1,
    value=itime0,
    description="time index",
    continuous_update=False,
)

slider_pol = SelectionSlider(
    options=POLARISATIONS,
    value=FIELD0,
    description="polarisation",
    continuous_update=False,
)

slider_x = IntSlider(
    min=0,
    max=N_spatial["x"] - 1,
    step=1,
    value=ix0,
    description="x index",
    continuous_update=False,
)

slider_y = IntSlider(
    min=0,
    max=N_spatial["y"] - 1,
    step=1,
    value=iy0,
    description="y index",
    continuous_update=False,
)

slider_z = IntSlider(
    min=0,
    max=N_spatial["z"] - 1,
    step=1,
    value=iz0,
    description="z index",
    continuous_update=False,
)

# =========================================================
# UPDATE FUNCTION
# =========================================================

def update_plot(change=None):

    itime = slider_time.value
    FIELD = slider_pol.value

    ix = slider_x.value
    iy = slider_y.value
    iz = slider_z.value

    t = timesteps[itime]

    # -----------------------------------------------------
    # Load current planes
    # -----------------------------------------------------

    field_xy, z_coord = load_plane(
        FIELD=FIELD,
        t=t,
        normal_axis="z",
        normal_index=iz,
    )

    field_xz, y_coord = load_plane(
        FIELD=FIELD,
        t=t,
        normal_axis="y",
        normal_index=iy,
    )

    field_yz, x_coord = load_plane(
        FIELD=FIELD,
        t=t,
        normal_axis="x",
        normal_index=ix,
    )

    # -----------------------------------------------------
    # Colour limits
    # -----------------------------------------------------

    vmin_xy, vmax_xy = current_limits(field_xy)
    vmin_xz, vmax_xz = current_limits(field_xz)
    vmin_yz, vmax_yz = current_limits(field_yz)

    # -----------------------------------------------------
    # Update images
    # -----------------------------------------------------

    im_xy.set_data(field_xy.T)
    im_xy.set_clim(vmin_xy, vmax_xy)

    im_xz.set_data(field_xz.T)
    im_xz.set_clim(vmin_xz, vmax_xz)

    im_yz.set_data(field_yz.T)
    im_yz.set_clim(vmin_yz, vmax_yz)

    # -----------------------------------------------------
    # Update titles
    # -----------------------------------------------------

    fig.suptitle(
        f"{FIELD} | t index = {itime}, t = {t} | "
        f"colourbar mode = {COLORBAR_MODE}",
        fontsize=14,
    )

    axs[0].set_title(f"xy plane | z = {z_coord:.4g}")
    axs[1].set_title(f"xz plane | y = {y_coord:.4g}")
    axs[2].set_title(f"yz plane | x = {x_coord:.4g}")

    # -----------------------------------------------------
    # Update colourbars
    # -----------------------------------------------------

    cbar_xy.update_normal(im_xy)
    cbar_xz.update_normal(im_xz)
    cbar_yz.update_normal(im_yz)

    cbar_xy.set_label(FIELD)
    cbar_xz.set_label(FIELD)
    cbar_yz.set_label(FIELD)

    fig.canvas.draw_idle()

# =========================================================
# CONNECT SLIDERS
# =========================================================

for slider in [
    slider_time,
    slider_pol,
    slider_x,
    slider_y,
    slider_z,
]:
    slider.observe(update_plot, names="value")

# =========================================================
# DISPLAY
# =========================================================

controls = VBox(
    [
        HBox([slider_time, slider_pol]),
        HBox([slider_x, slider_y, slider_z]),
    ]
)

display(
    VBox(
        [
            controls,
            fig.canvas,
        ]
    )
)

plt.ion()

In [ ]:
%matplotlib widget

import happi
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import IntSlider, SelectionSlider, ToggleButtons, VBox, HBox
from IPython.display import display

# =========================================================
# USER OPTIONS
# =========================================================

COLORBAR_MODE = "global"  # "global" or "adaptive"

INITIAL_FIELD_FAMILY = "E"   # "E" or "B"
INITIAL_POL_COMPONENT = "y"  # "x", "y", or "z"

cmap = "RdBu_r"

AXES = ["x", "y", "z"]

# Use 1 for a true global scan over all available timesteps.
TIMESTEP_STRIDE = 1

# Fallback only used if the data are exactly zero or invalid.
VMAX_FALLBACK = 1.0

# =========================================================
# BASIC FIELD SETUP
# =========================================================

COLORBAR_MODE = COLORBAR_MODE.lower()
INITIAL_FIELD_FAMILY = INITIAL_FIELD_FAMILY.upper()
INITIAL_POL_COMPONENT = INITIAL_POL_COMPONENT.lower()

FIELD_FAMILIES = ["E", "B"]
POL_COMPONENTS = ["x", "y", "z"]

if COLORBAR_MODE not in ["global", "adaptive"]:
    raise ValueError("COLORBAR_MODE must be either 'global' or 'adaptive'.")

if INITIAL_FIELD_FAMILY not in FIELD_FAMILIES:
    raise ValueError("INITIAL_FIELD_FAMILY must be either 'E' or 'B'.")

if INITIAL_POL_COMPONENT not in POL_COMPONENTS:
    raise ValueError("INITIAL_POL_COMPONENT must be 'x', 'y', or 'z'.")


def make_field_name(field_family, pol_component):
    """
    Construct Smilei field name from field family and component.

    Examples:
        "E", "y" -> "Ey"
        "B", "z" -> "Bz"
    """
    return f"{field_family.upper()}{pol_component.lower()}"


FIELD0 = make_field_name(
    INITIAL_FIELD_FAMILY,
    INITIAL_POL_COMPONENT,
)

# =========================================================
# OPEN SIMULATION
# =========================================================

S = happi.Open(path, verbose=False)

# =========================================================
# GEOMETRY
# =========================================================

ax_idx = {
    "x": 0,
    "y": 1,
    "z": 2,
}

N_spatial = {}
spatial_values = {}

for AXIS in AXES:

    L = S.namelist.Main.grid_length[ax_idx[AXIS]]
    d = S.namelist.Main.cell_length[ax_idx[AXIS]]

    N = int(round(L / d))

    N_spatial[AXIS] = N
    spatial_values[AXIS] = np.arange(N) * d

# =========================================================
# TIMESTEPS
# =========================================================

# Assumption:
# Ex, Ey, Ez, Bx, By, Bz share the same timestep list.
# Therefore we read the timesteps only once from the initial field.
F0 = S.Field(0, FIELD0)
timesteps = np.asarray(F0.getTimesteps())
timesteps = timesteps[::TIMESTEP_STRIDE]

if len(timesteps) == 0:
    raise RuntimeError("No timesteps available after applying TIMESTEP_STRIDE.")

# =========================================================
# HELPER FUNCTIONS
# =========================================================

def raw_absmax(data):
    """
    Return max(abs(data)) without imposing a fallback.
    """
    arr = np.asarray(data)

    if arr.size == 0:
        return 0.0

    abs_arr = np.abs(arr)

    if np.all(np.isnan(abs_arr)):
        return 0.0

    val = np.nanmax(abs_arr)

    if not np.isfinite(val):
        return 0.0

    return float(val)


def finalise_vmax(vmax):
    """
    Ensure colour scale is finite and non-zero.
    """
    if not np.isfinite(vmax) or vmax <= 0.0:
        return VMAX_FALLBACK

    return float(vmax)


def load_plane(FIELD, t, normal_axis, normal_index):
    """
    Load one 2D slice by fixing one coordinate.

    normal_axis = "z" gives xy plane
    normal_axis = "y" gives xz plane
    normal_axis = "x" gives yz plane
    """
    coord = spatial_values[normal_axis][normal_index]

    D = S.Field(
        0,
        FIELD,
        timesteps=t,
        subset={normal_axis: [coord]},
    )

    data = np.asarray(D.getData()[0])
    data = np.squeeze(data)

    return data, coord


def plane_extent(horizontal_axis, vertical_axis):
    """
    Return imshow extent for a selected pair of axes.
    """
    h = spatial_values[horizontal_axis]
    v = spatial_values[vertical_axis]

    return [
        float(h[0]),
        float(h[-1]),
        float(v[0]),
        float(v[-1]),
    ]


def adaptive_limits(data):
    """
    Symmetric colour limits for one subplot.
    """
    vmax = finalise_vmax(raw_absmax(data))
    return -vmax, vmax


# =========================================================
# GLOBAL MAXIMUM SCAN
# =========================================================

GLOBAL_VMAX_BY_FAMILY = {}

if COLORBAR_MODE == "global":

    print("Computing global normalisation...")
    print("Scanning E-fields and B-fields separately.")
    print(f"Number of timesteps used: {len(timesteps)}")

    for field_family in FIELD_FAMILIES:

        fields = [
            make_field_name(field_family, pol_component)
            for pol_component in POL_COMPONENTS
        ]

        print()
        print(f"Field family: {field_family}")
        print(f"Components: {fields}")

        vmax_scan = 0.0
        counter = 0
        total = len(fields) * len(timesteps)

        for FIELD in fields:

            for t in timesteps:

                counter += 1

                D = S.Field(
                    0,
                    FIELD,
                    timesteps=t,
                )

                data = D.getData()[0]

                vmax_scan = max(
                    vmax_scan,
                    raw_absmax(data),
                )

                print(
                    f"\rScanning {counter}/{total}: {FIELD}, t={t}",
                    end="",
                )

        GLOBAL_VMAX_BY_FAMILY[field_family] = finalise_vmax(vmax_scan)

        print()
        print(
            f"GLOBAL_VMAX[{field_family}] = "
            f"{GLOBAL_VMAX_BY_FAMILY[field_family]}"
        )

else:
    print("Adaptive colourbar mode selected.")
    print("No global scan will be performed.")


def current_limits(data, field_family):
    """
    Return symmetric colour limits according to COLORBAR_MODE.
    """
    if COLORBAR_MODE == "global":
        vmax = GLOBAL_VMAX_BY_FAMILY[field_family]
        return -vmax, vmax

    if COLORBAR_MODE == "adaptive":
        return adaptive_limits(data)

    raise ValueError("Unknown COLORBAR_MODE.")

# =========================================================
# INITIAL INDICES
# =========================================================

itime0 = 0

ix0 = N_spatial["x"] // 2
iy0 = N_spatial["y"] // 2
iz0 = N_spatial["z"] // 2

t0 = timesteps[itime0]

# =========================================================
# INITIAL DATA
# =========================================================

field_xy, z_coord = load_plane(
    FIELD=FIELD0,
    t=t0,
    normal_axis="z",
    normal_index=iz0,
)

field_xz, y_coord = load_plane(
    FIELD=FIELD0,
    t=t0,
    normal_axis="y",
    normal_index=iy0,
)

field_yz, x_coord = load_plane(
    FIELD=FIELD0,
    t=t0,
    normal_axis="x",
    normal_index=ix0,
)

vmin_xy, vmax_xy = current_limits(
    field_xy,
    INITIAL_FIELD_FAMILY,
)

vmin_xz, vmax_xz = current_limits(
    field_xz,
    INITIAL_FIELD_FAMILY,
)

vmin_yz, vmax_yz = current_limits(
    field_yz,
    INITIAL_FIELD_FAMILY,
)

# =========================================================
# FIGURE
# =========================================================

with plt.ioff():

    fig = plt.figure(
        figsize=(18, 6),
        constrained_layout=True,
    )

    gs = fig.add_gridspec(
        2,
        3,
        height_ratios=[1.0, 0.06],
    )

    axs = [
        fig.add_subplot(gs[0, 0]),
        fig.add_subplot(gs[0, 1]),
        fig.add_subplot(gs[0, 2]),
    ]

    caxs = [
        fig.add_subplot(gs[1, 0]),
        fig.add_subplot(gs[1, 1]),
        fig.add_subplot(gs[1, 2]),
    ]

    fig.suptitle(
        f"{FIELD0} | t index = {itime0}, t = {t0} | "
        f"colourbar mode = {COLORBAR_MODE}",
        fontsize=14,
    )

    # -----------------------------------------------------
    # xy plane, z fixed
    # -----------------------------------------------------

    im_xy = axs[0].imshow(
        field_xy.T,
        extent=plane_extent("x", "y"),
        origin="lower",
        aspect="auto",
        cmap=cmap,
        vmin=vmin_xy,
        vmax=vmax_xy,
    )

    axs[0].set_title(f"xy plane | z = {z_coord:.4g}")
    axs[0].set_xlabel("x")
    axs[0].set_ylabel("y")

    cbar_xy = fig.colorbar(
        im_xy,
        cax=caxs[0],
        orientation="horizontal",
    )
    cbar_xy.set_label(FIELD0)

    # -----------------------------------------------------
    # xz plane, y fixed
    # -----------------------------------------------------

    im_xz = axs[1].imshow(
        field_xz.T,
        extent=plane_extent("x", "z"),
        origin="lower",
        aspect="auto",
        cmap=cmap,
        vmin=vmin_xz,
        vmax=vmax_xz,
    )

    axs[1].set_title(f"xz plane | y = {y_coord:.4g}")
    axs[1].set_xlabel("x")
    axs[1].set_ylabel("z")

    cbar_xz = fig.colorbar(
        im_xz,
        cax=caxs[1],
        orientation="horizontal",
    )
    cbar_xz.set_label(FIELD0)

    # -----------------------------------------------------
    # yz plane, x fixed
    # -----------------------------------------------------

    im_yz = axs[2].imshow(
        field_yz.T,
        extent=plane_extent("y", "z"),
        origin="lower",
        aspect="auto",
        cmap=cmap,
        vmin=vmin_yz,
        vmax=vmax_yz,
    )

    axs[2].set_title(f"yz plane | x = {x_coord:.4g}")
    axs[2].set_xlabel("y")
    axs[2].set_ylabel("z")

    cbar_yz = fig.colorbar(
        im_yz,
        cax=caxs[2],
        orientation="horizontal",
    )
    cbar_yz.set_label(FIELD0)

# =========================================================
# WIDGETS
# =========================================================

widget_style = {
    "description_width": "initial",
}

slider_family = ToggleButtons(
    options=[
        ("E-field", "E"),
        ("B-field", "B"),
    ],
    value=INITIAL_FIELD_FAMILY,
    description="field",
    style=widget_style,
)

slider_pol = SelectionSlider(
    options=[
        ("x", "x"),
        ("y", "y"),
        ("z", "z"),
    ],
    value=INITIAL_POL_COMPONENT,
    description="polarisation",
    continuous_update=False,
    style=widget_style,
)

slider_time = IntSlider(
    min=0,
    max=len(timesteps) - 1,
    step=1,
    value=itime0,
    description="time index",
    continuous_update=False,
    style=widget_style,
)

slider_x = IntSlider(
    min=0,
    max=N_spatial["x"] - 1,
    step=1,
    value=ix0,
    description="x index",
    continuous_update=False,
    style=widget_style,
)

slider_y = IntSlider(
    min=0,
    max=N_spatial["y"] - 1,
    step=1,
    value=iy0,
    description="y index",
    continuous_update=False,
    style=widget_style,
)

slider_z = IntSlider(
    min=0,
    max=N_spatial["z"] - 1,
    step=1,
    value=iz0,
    description="z index",
    continuous_update=False,
    style=widget_style,
)

# =========================================================
# UPDATE FUNCTION
# =========================================================

def update_plot(change=None):

    field_family = slider_family.value
    pol_component = slider_pol.value

    FIELD = make_field_name(
        field_family,
        pol_component,
    )

    itime = slider_time.value
    ix = slider_x.value
    iy = slider_y.value
    iz = slider_z.value

    t = timesteps[itime]

    # -----------------------------------------------------
    # Load current planes
    # -----------------------------------------------------

    field_xy, z_coord = load_plane(
        FIELD=FIELD,
        t=t,
        normal_axis="z",
        normal_index=iz,
    )

    field_xz, y_coord = load_plane(
        FIELD=FIELD,
        t=t,
        normal_axis="y",
        normal_index=iy,
    )

    field_yz, x_coord = load_plane(
        FIELD=FIELD,
        t=t,
        normal_axis="x",
        normal_index=ix,
    )

    # -----------------------------------------------------
    # Colour limits
    # -----------------------------------------------------

    vmin_xy, vmax_xy = current_limits(
        field_xy,
        field_family,
    )

    vmin_xz, vmax_xz = current_limits(
        field_xz,
        field_family,
    )

    vmin_yz, vmax_yz = current_limits(
        field_yz,
        field_family,
    )

    # -----------------------------------------------------
    # Update image data and colour limits
    # -----------------------------------------------------

    im_xy.set_data(field_xy.T)
    im_xy.set_clim(vmin_xy, vmax_xy)

    im_xz.set_data(field_xz.T)
    im_xz.set_clim(vmin_xz, vmax_xz)

    im_yz.set_data(field_yz.T)
    im_yz.set_clim(vmin_yz, vmax_yz)

    # -----------------------------------------------------
    # Update titles
    # -----------------------------------------------------

    fig.suptitle(
        f"{FIELD} | t index = {itime}, t = {t} | "
        f"colourbar mode = {COLORBAR_MODE}",
        fontsize=14,
    )

    axs[0].set_title(f"xy plane | z = {z_coord:.4g}")
    axs[1].set_title(f"xz plane | y = {y_coord:.4g}")
    axs[2].set_title(f"yz plane | x = {x_coord:.4g}")

    # -----------------------------------------------------
    # Update colourbars
    # -----------------------------------------------------

    cbar_xy.update_normal(im_xy)
    cbar_xz.update_normal(im_xz)
    cbar_yz.update_normal(im_yz)

    cbar_xy.set_label(FIELD)
    cbar_xz.set_label(FIELD)
    cbar_yz.set_label(FIELD)

    fig.canvas.draw_idle()

# =========================================================
# CONNECT WIDGETS
# =========================================================

for widget in [
    slider_family,
    slider_pol,
    slider_time,
    slider_x,
    slider_y,
    slider_z,
]:
    widget.observe(update_plot, names="value")

# =========================================================
# DISPLAY
# =========================================================

controls = VBox(
    [
        HBox([slider_family, slider_pol, slider_time]),
        HBox([slider_x, slider_y, slider_z]),
    ]
)

display(
    VBox(
        [
            controls,
            fig.canvas,
        ]
    )
)

In [ ]:
%matplotlib widget

import happi
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import IntSlider, SelectionSlider, ToggleButtons, VBox, HBox
from IPython.display import display

# =========================================================
# USER OPTIONS
# =========================================================

INITIAL_FIELD_FAMILY = "E"      # "E" or "B"
INITIAL_POL_COMPONENT = "y"     # "x", "y", or "z"
INITIAL_NORMALISATION = "global"  # "global" or "adaptive"

cmap = "RdBu_r"

AXES = ["x", "y", "z"]

# Use 1 for a true global scan over all available timesteps.
# If this is increased, both plotting and global scans use the reduced timestep list.
TIMESTEP_STRIDE = 1

# Fallback only used if the data are exactly zero or invalid.
VMAX_FALLBACK = 1.0

# =========================================================
# BASIC FIELD SETUP
# =========================================================

INITIAL_FIELD_FAMILY = INITIAL_FIELD_FAMILY.upper()
INITIAL_POL_COMPONENT = INITIAL_POL_COMPONENT.lower()
INITIAL_NORMALISATION = INITIAL_NORMALISATION.lower()

FIELD_FAMILIES = ["E", "B"]
POL_COMPONENTS = ["x", "y", "z"]
NORMALISATION_MODES = ["global", "adaptive"]

if INITIAL_FIELD_FAMILY not in FIELD_FAMILIES:
    raise ValueError("INITIAL_FIELD_FAMILY must be either 'E' or 'B'.")

if INITIAL_POL_COMPONENT not in POL_COMPONENTS:
    raise ValueError("INITIAL_POL_COMPONENT must be 'x', 'y', or 'z'.")

if INITIAL_NORMALISATION not in NORMALISATION_MODES:
    raise ValueError("INITIAL_NORMALISATION must be 'global' or 'adaptive'.")


def make_field_name(field_family, pol_component):
    """
    Construct Smilei field name from field family and component.

    Examples:
        "E", "y" -> "Ey"
        "B", "z" -> "Bz"
    """
    return f"{field_family.upper()}{pol_component.lower()}"


FIELD0 = make_field_name(
    INITIAL_FIELD_FAMILY,
    INITIAL_POL_COMPONENT,
)

# =========================================================
# OPEN SIMULATION
# =========================================================

S = happi.Open(path, verbose=False)

# =========================================================
# GEOMETRY
# =========================================================

ax_idx = {
    "x": 0,
    "y": 1,
    "z": 2,
}

N_spatial = {}
spatial_values = {}

for AXIS in AXES:

    L = S.namelist.Main.grid_length[ax_idx[AXIS]]
    d = S.namelist.Main.cell_length[ax_idx[AXIS]]

    N = int(round(L / d))

    N_spatial[AXIS] = N
    spatial_values[AXIS] = np.arange(N) * d

# =========================================================
# TIMESTEPS
# =========================================================

# Assumption:
# Ex, Ey, Ez, Bx, By, Bz share the same timestep list.
# Therefore we read the timesteps only once from the initial field.
F0 = S.Field(0, FIELD0)
timesteps = np.asarray(F0.getTimesteps())
timesteps = timesteps[::TIMESTEP_STRIDE]

if len(timesteps) == 0:
    raise RuntimeError("No timesteps available after applying TIMESTEP_STRIDE.")

# =========================================================
# HELPER FUNCTIONS
# =========================================================

def raw_absmax(data):
    """
    Return max(abs(data)) without imposing a fallback.
    """
    arr = np.asarray(data)

    if arr.size == 0:
        return 0.0

    abs_arr = np.abs(arr)

    if np.all(np.isnan(abs_arr)):
        return 0.0

    val = np.nanmax(abs_arr)

    if not np.isfinite(val):
        return 0.0

    return float(val)


def finalise_vmax(vmax):
    """
    Ensure colour scale is finite and non-zero.
    """
    if not np.isfinite(vmax) or vmax <= 0.0:
        return VMAX_FALLBACK

    return float(vmax)


def load_plane(FIELD, t, normal_axis, normal_index):
    """
    Load one 2D slice by fixing one coordinate.

    normal_axis = "z" gives xy plane
    normal_axis = "y" gives xz plane
    normal_axis = "x" gives yz plane
    """
    coord = spatial_values[normal_axis][normal_index]

    D = S.Field(
        0,
        FIELD,
        timesteps=t,
        subset={normal_axis: [coord]},
    )

    data = np.asarray(D.getData()[0])
    data = np.squeeze(data)

    return data, coord


def plane_extent(horizontal_axis, vertical_axis):
    """
    Return imshow extent for a selected pair of axes.
    """
    h = spatial_values[horizontal_axis]
    v = spatial_values[vertical_axis]

    return [
        float(h[0]),
        float(h[-1]),
        float(v[0]),
        float(v[-1]),
    ]


def adaptive_limits(data):
    """
    Symmetric colour limits for one subplot.
    """
    vmax = finalise_vmax(raw_absmax(data))
    return -vmax, vmax


def current_limits(data, field_family, normalisation):
    """
    Return symmetric colour limits according to the selected normalisation mode.

    global:
        use one precomputed maximum for the selected E/B field family

    adaptive:
        use the current subplot's own maximum
    """
    if normalisation == "global":
        vmax = GLOBAL_VMAX_BY_FAMILY[field_family]
        return -vmax, vmax

    if normalisation == "adaptive":
        return adaptive_limits(data)

    raise ValueError("Unknown normalisation mode.")

# =========================================================
# GLOBAL MAXIMUM SCAN
# =========================================================

# Since normalisation is now interactive, we precompute both global maxima
# before plotting. This makes switching from adaptive to global immediate.
GLOBAL_VMAX_BY_FAMILY = {}

print("Computing global normalisation for interactive global mode...")
print("Scanning E-fields and B-fields separately.")
print(f"Number of timesteps used: {len(timesteps)}")

for field_family in FIELD_FAMILIES:

    fields = [
        make_field_name(field_family, pol_component)
        for pol_component in POL_COMPONENTS
    ]

    print()
    print(f"Field family: {field_family}")
    print(f"Components: {fields}")

    vmax_scan = 0.0
    counter = 0
    total = len(fields) * len(timesteps)

    for FIELD in fields:

        for t in timesteps:

            counter += 1

            D = S.Field(
                0,
                FIELD,
                timesteps=t,
            )

            data = D.getData()[0]

            vmax_scan = max(
                vmax_scan,
                raw_absmax(data),
            )

            print(
                f"\rScanning {counter}/{total}: {FIELD}, t={t}",
                end="",
            )

    GLOBAL_VMAX_BY_FAMILY[field_family] = finalise_vmax(vmax_scan)

    print()
    print(
        f"GLOBAL_VMAX[{field_family}] = "
        f"{GLOBAL_VMAX_BY_FAMILY[field_family]}"
    )

# =========================================================
# INITIAL INDICES
# =========================================================

itime0 = 0

ix0 = N_spatial["x"] // 2
iy0 = N_spatial["y"] // 2
iz0 = N_spatial["z"] // 2

t0 = timesteps[itime0]

# =========================================================
# INITIAL DATA
# =========================================================

field_xy, z_coord = load_plane(
    FIELD=FIELD0,
    t=t0,
    normal_axis="z",
    normal_index=iz0,
)

field_xz, y_coord = load_plane(
    FIELD=FIELD0,
    t=t0,
    normal_axis="y",
    normal_index=iy0,
)

field_yz, x_coord = load_plane(
    FIELD=FIELD0,
    t=t0,
    normal_axis="x",
    normal_index=ix0,
)

vmin_xy, vmax_xy = current_limits(
    field_xy,
    INITIAL_FIELD_FAMILY,
    INITIAL_NORMALISATION,
)

vmin_xz, vmax_xz = current_limits(
    field_xz,
    INITIAL_FIELD_FAMILY,
    INITIAL_NORMALISATION,
)

vmin_yz, vmax_yz = current_limits(
    field_yz,
    INITIAL_FIELD_FAMILY,
    INITIAL_NORMALISATION,
)

# =========================================================
# FIGURE
# =========================================================

with plt.ioff():

    fig = plt.figure(
        figsize=(18, 6),
        constrained_layout=True,
    )

    gs = fig.add_gridspec(
        2,
        3,
        height_ratios=[1.0, 0.06],
    )

    axs = [
        fig.add_subplot(gs[0, 0]),
        fig.add_subplot(gs[0, 1]),
        fig.add_subplot(gs[0, 2]),
    ]

    caxs = [
        fig.add_subplot(gs[1, 0]),
        fig.add_subplot(gs[1, 1]),
        fig.add_subplot(gs[1, 2]),
    ]

    fig.suptitle(
        f"{FIELD0} | t index = {itime0}, t = {t0} | "
        f"normalisation = {INITIAL_NORMALISATION}",
        fontsize=14,
    )

    # -----------------------------------------------------
    # xy plane, z fixed
    # -----------------------------------------------------

    im_xy = axs[0].imshow(
        field_xy.T,
        extent=plane_extent("x", "y"),
        origin="lower",
        aspect="auto",
        cmap=cmap,
        vmin=vmin_xy,
        vmax=vmax_xy,
    )

    axs[0].set_title(f"xy plane | z = {z_coord:.4g}")
    axs[0].set_xlabel("x")
    axs[0].set_ylabel("y")

    cbar_xy = fig.colorbar(
        im_xy,
        cax=caxs[0],
        orientation="horizontal",
    )
    cbar_xy.set_label(FIELD0)

    # -----------------------------------------------------
    # xz plane, y fixed
    # -----------------------------------------------------

    im_xz = axs[1].imshow(
        field_xz.T,
        extent=plane_extent("x", "z"),
        origin="lower",
        aspect="auto",
        cmap=cmap,
        vmin=vmin_xz,
        vmax=vmax_xz,
    )

    axs[1].set_title(f"xz plane | y = {y_coord:.4g}")
    axs[1].set_xlabel("x")
    axs[1].set_ylabel("z")

    cbar_xz = fig.colorbar(
        im_xz,
        cax=caxs[1],
        orientation="horizontal",
    )
    cbar_xz.set_label(FIELD0)

    # -----------------------------------------------------
    # yz plane, x fixed
    # -----------------------------------------------------

    im_yz = axs[2].imshow(
        field_yz.T,
        extent=plane_extent("y", "z"),
        origin="lower",
        aspect="auto",
        cmap=cmap,
        vmin=vmin_yz,
        vmax=vmax_yz,
    )

    axs[2].set_title(f"yz plane | x = {x_coord:.4g}")
    axs[2].set_xlabel("y")
    axs[2].set_ylabel("z")

    cbar_yz = fig.colorbar(
        im_yz,
        cax=caxs[2],
        orientation="horizontal",
    )
    cbar_yz.set_label(FIELD0)

# =========================================================
# WIDGETS
# =========================================================

widget_style = {
    "description_width": "initial",
}

slider_family = ToggleButtons(
    options=[
        ("E-field", "E"),
        ("B-field", "B"),
    ],
    value=INITIAL_FIELD_FAMILY,
    description="field",
    style=widget_style,
)

slider_pol = SelectionSlider(
    options=[
        ("x", "x"),
        ("y", "y"),
        ("z", "z"),
    ],
    value=INITIAL_POL_COMPONENT,
    description="polarisation",
    continuous_update=False,
    style=widget_style,
)

slider_norm = ToggleButtons(
    options=[
        ("global", "global"),
        ("adaptive", "adaptive"),
    ],
    value=INITIAL_NORMALISATION,
    description="normalisation",
    style=widget_style,
)

slider_time = IntSlider(
    min=0,
    max=len(timesteps) - 1,
    step=1,
    value=itime0,
    description="time index",
    continuous_update=False,
    style=widget_style,
)

slider_x = IntSlider(
    min=0,
    max=N_spatial["x"] - 1,
    step=1,
    value=ix0,
    description="x index",
    continuous_update=False,
    style=widget_style,
)

slider_y = IntSlider(
    min=0,
    max=N_spatial["y"] - 1,
    step=1,
    value=iy0,
    description="y index",
    continuous_update=False,
    style=widget_style,
)

slider_z = IntSlider(
    min=0,
    max=N_spatial["z"] - 1,
    step=1,
    value=iz0,
    description="z index",
    continuous_update=False,
    style=widget_style,
)

# =========================================================
# UPDATE FUNCTION
# =========================================================

def update_plot(change=None):

    field_family = slider_family.value
    pol_component = slider_pol.value
    normalisation = slider_norm.value

    FIELD = make_field_name(
        field_family,
        pol_component,
    )

    itime = slider_time.value
    ix = slider_x.value
    iy = slider_y.value
    iz = slider_z.value

    t = timesteps[itime]

    # -----------------------------------------------------
    # Load current planes
    # -----------------------------------------------------

    field_xy, z_coord = load_plane(
        FIELD=FIELD,
        t=t,
        normal_axis="z",
        normal_index=iz,
    )

    field_xz, y_coord = load_plane(
        FIELD=FIELD,
        t=t,
        normal_axis="y",
        normal_index=iy,
    )

    field_yz, x_coord = load_plane(
        FIELD=FIELD,
        t=t,
        normal_axis="x",
        normal_index=ix,
    )

    # -----------------------------------------------------
    # Colour limits
    # -----------------------------------------------------

    vmin_xy, vmax_xy = current_limits(
        field_xy,
        field_family,
        normalisation,
    )

    vmin_xz, vmax_xz = current_limits(
        field_xz,
        field_family,
        normalisation,
    )

    vmin_yz, vmax_yz = current_limits(
        field_yz,
        field_family,
        normalisation,
    )

    # -----------------------------------------------------
    # Update image data and colour limits
    # -----------------------------------------------------

    im_xy.set_data(field_xy.T)
    im_xy.set_clim(vmin_xy, vmax_xy)

    im_xz.set_data(field_xz.T)
    im_xz.set_clim(vmin_xz, vmax_xz)

    im_yz.set_data(field_yz.T)
    im_yz.set_clim(vmin_yz, vmax_yz)

    # -----------------------------------------------------
    # Update titles
    # -----------------------------------------------------

    fig.suptitle(
        f"{FIELD} | t index = {itime}, t = {t} | "
        f"normalisation = {normalisation}",
        fontsize=14,
    )

    axs[0].set_title(f"xy plane | z = {z_coord:.4g}")
    axs[1].set_title(f"xz plane | y = {y_coord:.4g}")
    axs[2].set_title(f"yz plane | x = {x_coord:.4g}")

    # -----------------------------------------------------
    # Update colourbars
    # -----------------------------------------------------

    cbar_xy.update_normal(im_xy)
    cbar_xz.update_normal(im_xz)
    cbar_yz.update_normal(im_yz)

    cbar_xy.set_label(FIELD)
    cbar_xz.set_label(FIELD)
    cbar_yz.set_label(FIELD)

    fig.canvas.draw_idle()

# =========================================================
# CONNECT WIDGETS
# =========================================================

for widget in [
    slider_family,
    slider_pol,
    slider_norm,
    slider_time,
    slider_x,
    slider_y,
    slider_z,
]:
    widget.observe(update_plot, names="value")

# =========================================================
# DISPLAY
# =========================================================

controls = VBox(
    [
        HBox([slider_family, slider_pol, slider_norm]),
        HBox([slider_time]),
        HBox([slider_x, slider_y, slider_z]),
    ]
)

display(
    VBox(
        [
            controls,
            fig.canvas,
        ]
    )
)